# Food Calorie Classification Model
Notebook ini digunakan untuk melatih model klasifikasi gambar makanan dan mengestimasi kalorinya menggunakan arsitektur EfficientNetB0.

### 1. Install Dependencies
Menginstal library yang dibutuhkan.

In [ ]:
!pip install tensorflow matplotlib numpy

### 2. Import Libraries
Memuat modul-modul dasar dari TensorFlow, Keras, dan Matplotlib.

In [ ]:
import os
import json
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, applications
import matplotlib.pyplot as plt
import numpy as np

print(f"TensorFlow Version: {tf.__version__}")

### 3. Configuration & Calorie Mapping
Mengatur parameter training dan memuat kamus kalori dari file JSON eksternal.

In [ ]:
DATASET_PATH = "Dataset"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 40

try:
    with open("calorie_dict.json", "r") as f:
        calorie_dict = json.load(f)
    print("Calorie dictionary loaded successfully.")
except FileNotFoundError:
    print("Error: calorie_dict.json not found.")
    calorie_dict = {}

### 4. Load Datasets
Membaca dataset gambar dari folder dan membaginya menjadi data training dan validation.

In [ ]:
train_dataset = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_dataset.class_names
print("\nDetected Classes:", class_names)
NUM_CLASSES = len(class_names)

### 5. Data Augmentation
Menerapkan augmentasi gambar secara acak untuk memperkuat model terhadap variasi foto.

In [ ]:
data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.2),
        layers.RandomZoom(0.2),
        layers.RandomContrast(0.2),
        layers.RandomBrightness(0.2),
    ],
    name="data_augmentation"
)

plt.figure(figsize=(10, 10))
for images, _ in train_dataset.take(1):
    for i in range(9):
        augmented_images = data_augmentation(images)
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(augmented_images[0].numpy().astype("uint8"))
        plt.axis("off")
plt.suptitle("Data Augmentation Preview")
plt.show()

### 6. Pipeline Optimization
Mengoptimalkan performa I/O pipeline data selama proses training.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.prefetch(buffer_size=AUTOTUNE)

### 7. Model Architecture (EfficientNetB0)
Membangun arsitektur Transfer Learning menggunakan EfficientNetB0 dengan Custom Classifier.

In [ ]:
def build_model(num_classes):
    base_model = applications.EfficientNetB0(
        input_shape=IMG_SIZE + (3,),
        include_top=False,
        weights='imagenet'
    )
    
    base_model.trainable = True
    for layer in base_model.layers[:-30]:
        layer.trainable = False

    inputs = keras.Input(shape=IMG_SIZE + (3,))
    x = data_augmentation(inputs)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-4),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=['accuracy']
    )
    return model, base_model

model, base_model = build_model(NUM_CLASSES)
model.summary()

### 8. Training
Memulai proses pelatihan model dengan strategi EarlyStopping dan ModelCheckpoint.

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6),
    keras.callbacks.ModelCheckpoint('best_food_model.keras', save_best_only=True)
]

history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=val_dataset,
    callbacks=callbacks
)

### 9. Evaluation
Memvisualisasikan grafik akurasi dan loss dari proses training.

In [ ]:
def plot_history(history):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs_range = range(len(acc))

    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label='Training Accuracy')
    plt.plot(epochs_range, val_acc, label='Validation Accuracy')
    plt.legend(loc='lower right')
    plt.title('Training and Validation Accuracy')

    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Training Loss')
    plt.plot(epochs_range, val_loss, label='Validation Loss')
    plt.legend(loc='upper right')
    plt.title('Training and Validation Loss')
    plt.show()

plot_history(history)

### 10. Prediction Function
Fungsi utilitas untuk memprediksi foto makanan secara langsung.

In [ ]:
def predict_food_calorie(img_path):
    img = keras.preprocessing.image.load_img(img_path, target_size=IMG_SIZE)
    img_array = keras.preprocessing.image.img_to_array(img)
    img_array = tf.expand_dims(img_array, 0)

    predictions = model.predict(img_array)
    score = tf.nn.softmax(predictions[0])
    
    predicted_class = class_names[np.argmax(score)]
    confidence = 100 * np.max(score)
    estimated_calories = calorie_dict.get(predicted_class, "Unknown")
    
    plt.figure(figsize=(6,6))
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"Class: {predicted_class} ({confidence:.2f}%)\nCalories: {estimated_calories} kcal")
    plt.show()

# predict_food_calorie("Dataset/Apel/contoh_apel.jpg")

### 11. Export Final Model
Menyimpan keadaan model di epoch paling terakhir.

In [ ]:
model.save('food_calorie_model_final.keras')